# Rolling PCA model-selection notebook for swaption surface hedging

This notebook is designed to help you choose:

- the **best rolling window length**,
- the **best rolling step**,
- the **best subset of hedge tenors** to keep as regression inputs,

for a PCA-based hedge reduction framework on a rates or swaption surface.

The notebook is built around your current codebase:

- `market_data.py`
- `covariance_estimators.py`
- `pca_engines.py`

and adds a **notebook-local regression layer** so the selection workflow is fully testable even if the regression class is not yet integrated into `pca_engines.py`.

## Scope and philosophy

At this stage, you do **not** yet have desk sensitivities, so the selection criterion is not real hedge PnL yet.

For now, the notebook selects configurations that jointly optimize:

1. **PCA structural quality**
   - mean and downside explained variance of the top `K` PCs,
   - loading stability across rolling windows,

2. **Tradable replication quality**
   - how well selected hedge tenors can replicate PCA scores,
   - how well they can reconstruct the full processed surface,

3. **Operational robustness**
   - stability / turnover of the reduced hedge mapping across rolling windows.

The notebook is intentionally **in-sample rolling**. It is meant as a first model-selection pass.  
Once you have narrowed the search space, the next step should be a **pseudo out-of-sample** test and later a **real PnL hedge test** with desk sensitivities.

In [ ]:
import os
import sys
import itertools
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.linear_model import LinearRegression, Ridge, Lasso

CURRENT_WORKING_DIRECTORY = os.getcwd()
if CURRENT_WORKING_DIRECTORY not in sys.path:
    sys.path.append(CURRENT_WORKING_DIRECTORY)

from market_data import CurveInfo, MarketData
from covariance_estimators import CovarianceEstimator
from pca_engines import StaticPCA, RollingPCA

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")

## User parameters

Edit this cell first.

A few practical comments:

- Keep `standardize=False` at first. It is usually better to start from raw daily normal-vol changes after demeaning.
- Set the EWMA half-life **explicitly**. Do not rely on the default fallback.
- Keep the candidate pool intentionally small at first. A spread of 10 to 20 nodes across the surface is enough for a first pass.

In [ ]:
# -------------------------------------------------------------------
# Data specification
# -------------------------------------------------------------------
curve_info = CurveInfo(
    index="SOFR",
    ccy="USD",
    curve_id="N0",
    closing="PARIS",
    is_rate=False
)

source = "SUMMIT"
start_date = "15/01/25"
end_date = "02/03/26"

rates_tenors_universe = [
    "2Y", "3Y", "4Y", "5Y", "6Y", "7Y", "8Y", "9Y", "10Y", "11Y",
    "12Y", "13Y", "14Y", "15Y", "20Y", "25Y", "30Y", "40Y", "50Y"
]

vol_expiries_universe = [
    "1M", "3M", "6M", "1Y", "18M", "2Y", "3Y", "4Y", "5Y",
    "7Y", "10Y", "12Y", "15Y", "20Y", "25Y", "30Y"
]

vol_tenors_universe = [
    "1Y", "2Y", "3Y", "4Y", "5Y", "6Y", "7Y", "8Y", "9Y",
    "10Y", "15Y", "20Y", "25Y", "30Y"
]

# -------------------------------------------------------------------
# PCA preprocessing
# -------------------------------------------------------------------
compute_changes = True
demean = True
standardize = False
number_of_components = 3

# -------------------------------------------------------------------
# Covariance estimator
# -------------------------------------------------------------------
covariance_method = "ewma"
ewma_halflife = 60
alpha_ewma = None

# -------------------------------------------------------------------
# Rolling search grid
# -------------------------------------------------------------------
window_grid = ["3M", "6M", "9M", "12M"]
step_grid = ["1D", "1W", "2W", "1M"]

alignment_anchor = "previous"

# -------------------------------------------------------------------
# Regression search
# -------------------------------------------------------------------
regression_type = "ridge"       # "ols", "ridge", or "lasso"
ridge_or_lasso_alpha = 1.0
fit_intercept = False
lasso_max_iterations = 20000

# -------------------------------------------------------------------
# Candidate hedge-tenor pool
# Edit this list to reflect the actual tradable nodes you care about.
# -------------------------------------------------------------------
candidate_pool = [
    "3M2Y", "3M10Y",
    "1Y5Y", "1Y10Y",
    "2Y2Y", "2Y10Y",
    "5Y5Y", "5Y10Y", "5Y20Y",
    "10Y5Y", "10Y10Y", "10Y20Y",
    "20Y10Y", "20Y20Y"
]

subset_sizes_to_test = [2, 3, 4, 5, 6]
number_of_window_step_pairs_to_shortlist = 3

# -------------------------------------------------------------------
# Minimum admissibility thresholds for rolling PCA structure
# These are not sacred. They are just sensible first-pass filters.
# -------------------------------------------------------------------
minimum_mean_pc1_stability = 0.98
minimum_mean_pc2_stability = 0.97
minimum_p10_topk_explained_variance = 90.0

## Load and preprocess the data

In [ ]:
market_data = MarketData(
    curve_info=curve_info,
    source=source,
    start_date=start_date,
    end_date=end_date,
    rates_tenors_universe=rates_tenors_universe,
    vol_tenors_universe=vol_tenors_universe,
    vol_expiries_universe=vol_expiries_universe
)

market_data.load()
market_data.preprocess(
    compute_changes=compute_changes,
    demean=demean,
    standardize=standardize
)

available_columns = list(market_data.processed_data.columns)
candidate_pool = [tenor for tenor in candidate_pool if tenor in available_columns]

print("Raw data shape:", market_data.data.shape)
print("Processed data shape:", market_data.processed_data.shape)
print("Start / end:", market_data.processed_data.index.min(), market_data.processed_data.index.max())
print("Number of candidate hedge tenors kept:", len(candidate_pool))
print("Candidate hedge tenors:", candidate_pool)

display(market_data.data.head())
display(market_data.processed_data.head())

## Helper functions

This cell contains:

- PCA stability diagnostics,
- a notebook-local regression layer from selected tenors to PCA scores,
- rolling evaluation helpers,
- greedy forward tenor selection,
- visualization utilities.

The code is explicit on purpose so it is easy to inspect and modify.

In [ ]:
# -------------------------------------------------------------------
# Parsing helpers
# -------------------------------------------------------------------
def maturity_to_years(maturity_label: str) -> float:
    maturity_label = str(maturity_label).strip().upper()

    if maturity_label.endswith("M"):
        return float(maturity_label[:-1]) / 12.0

    if maturity_label.endswith("Y"):
        return float(maturity_label[:-1])

    raise ValueError(f"Unknown maturity format: {maturity_label}")


def parse_expiry_tenor_label(surface_label: str) -> Tuple[str, str]:
    surface_label = str(surface_label).strip().upper()

    for split_position in range(2, len(surface_label)):
        left_label = surface_label[:split_position]
        right_label = surface_label[split_position:]

        if left_label[-1] in ("M", "Y") and right_label.endswith("Y"):
            return left_label, right_label

    raise ValueError(f"Could not parse expiry/tenor label: {surface_label}")


def sort_maturity_labels(maturity_labels: List[str]) -> List[str]:
    return sorted(maturity_labels, key=maturity_to_years)


# -------------------------------------------------------------------
# Metrics
# -------------------------------------------------------------------
def compute_reconstruction_metrics(actual_frame: pd.DataFrame, residual_frame: pd.DataFrame) -> Dict[str, object]:
    squared_residuals = np.square(residual_frame.values)
    absolute_residuals = np.abs(residual_frame.values)

    mse_global = float(np.mean(squared_residuals))
    rmse_global = float(np.sqrt(mse_global))
    mae_global = float(np.mean(absolute_residuals))

    actual_mean_by_node = actual_frame.mean(axis=0)
    sum_squared_errors = float(np.sum(squared_residuals))
    total_sum_of_squares = float(np.sum(np.square((actual_frame - actual_mean_by_node).values)))
    r2_global = 1.0 - sum_squared_errors / total_sum_of_squares if total_sum_of_squares > 0 else np.nan

    rmse_by_node = pd.Series(
        np.sqrt(np.mean(squared_residuals, axis=0)),
        index=actual_frame.columns,
        name="rmse_by_node"
    )

    mae_by_node = pd.Series(
        np.mean(absolute_residuals, axis=0),
        index=actual_frame.columns,
        name="mae_by_node"
    )

    rmse_by_date = pd.Series(
        np.sqrt(np.mean(squared_residuals, axis=1)),
        index=actual_frame.index,
        name="rmse_by_date"
    )

    mae_by_date = pd.Series(
        np.mean(absolute_residuals, axis=1),
        index=actual_frame.index,
        name="mae_by_date"
    )

    return {
        "RMSE Global": rmse_global,
        "MAE Global": mae_global,
        "R2": r2_global,
        "RMSE by node": rmse_by_node,
        "MAE by node": mae_by_node,
        "RMSE by date": rmse_by_date,
        "MAE by date": mae_by_date,
    }


def compute_score_fit_metrics(actual_scores: pd.DataFrame, predicted_scores: pd.DataFrame) -> Dict[str, object]:
    score_fit_rows = []

    for pc_name in actual_scores.columns:
        actual_values = actual_scores[pc_name].values
        predicted_values = predicted_scores[pc_name].values
        residual_values = actual_values - predicted_values

        mse_value = float(np.mean(np.square(residual_values)))
        rmse_value = float(np.sqrt(mse_value))
        mae_value = float(np.mean(np.abs(residual_values)))

        sum_squared_errors = float(np.sum(np.square(residual_values)))
        total_sum_of_squares = float(np.sum(np.square(actual_values - np.mean(actual_values))))
        r2_value = 1.0 - sum_squared_errors / total_sum_of_squares if total_sum_of_squares > 0 else np.nan

        score_fit_rows.append(
            {
                "pc": pc_name,
                "RMSE": rmse_value,
                "MAE": mae_value,
                "R2": r2_value,
            }
        )

    score_fit_frame = pd.DataFrame(score_fit_rows).set_index("pc")

    return {
        "summary": score_fit_frame,
        "mean_r2": float(score_fit_frame["R2"].mean()),
        "median_r2": float(score_fit_frame["R2"].median()),
        "min_r2": float(score_fit_frame["R2"].min()),
    }


# -------------------------------------------------------------------
# Rolling PCA structural diagnostics
# -------------------------------------------------------------------
def build_explained_variance_frame(rolling_pca: RollingPCA) -> pd.DataFrame:
    explained_variance_rows = []

    for rolling_window_pca in rolling_pca.pcas_results:
        window_end_date = rolling_window_pca.results.scores.index[-1]
        row = {"window_end": window_end_date}

        for component_index in range(rolling_pca.n_components):
            row[f"PC{component_index + 1}"] = float(
                rolling_window_pca.results.explained_var_ratios[component_index] * 100.0
            )

        row["TopK"] = float(
            np.sum(rolling_window_pca.results.explained_var_ratios[:rolling_pca.n_components]) * 100.0
        )
        explained_variance_rows.append(row)

    return pd.DataFrame(explained_variance_rows).set_index("window_end")


def build_loading_stability_frame(rolling_pca: RollingPCA) -> pd.DataFrame:
    if len(rolling_pca.pcas_results) <= 1:
        return pd.DataFrame()

    stability_rows = []

    for window_index in range(1, len(rolling_pca.pcas_results)):
        previous_pca = rolling_pca.pcas_results[window_index - 1]
        current_pca = rolling_pca.pcas_results[window_index]

        row = {"window_end": current_pca.results.scores.index[-1]}

        for component_index in range(rolling_pca.n_components):
            previous_loading = previous_pca.results.loadings.iloc[:, component_index].values
            current_loading = current_pca.results.loadings.iloc[:, component_index].values

            similarity = float(
                np.dot(previous_loading, current_loading)
                / (np.linalg.norm(previous_loading) * np.linalg.norm(current_loading))
            )

            row[f"PC{component_index + 1}"] = similarity

        stability_rows.append(row)

    return pd.DataFrame(stability_rows).set_index("window_end")


def evaluate_window_step_configuration(
    base_market_data: MarketData,
    window_size: str,
    step_size: str,
    covariance_estimator: CovarianceEstimator,
    number_of_components: int,
    alignment_anchor: str = "previous",
) -> Dict[str, object]:
    rolling_pca = RollingPCA(
        data=base_market_data,
        window_size=window_size,
        step=step_size,
        n_components=number_of_components,
        cov_estimator=covariance_estimator,
    )

    rolling_pca.fit()
    rolling_pca.align_components(anchor=alignment_anchor)

    explained_variance_frame = build_explained_variance_frame(rolling_pca)
    loading_stability_frame = build_loading_stability_frame(rolling_pca)

    summary = {
        "window_size": window_size,
        "step_size": step_size,
        "n_windows": len(rolling_pca.pcas_results),
        "mean_topk_explained_variance": float(explained_variance_frame["TopK"].mean()),
        "median_topk_explained_variance": float(explained_variance_frame["TopK"].median()),
        "p10_topk_explained_variance": float(explained_variance_frame["TopK"].quantile(0.10)),
        "min_topk_explained_variance": float(explained_variance_frame["TopK"].min()),
    }

    for component_index in range(number_of_components):
        component_name = f"PC{component_index + 1}"

        if not loading_stability_frame.empty:
            summary[f"mean_{component_name.lower()}_stability"] = float(loading_stability_frame[component_name].mean())
            summary[f"median_{component_name.lower()}_stability"] = float(loading_stability_frame[component_name].median())
            summary[f"p10_{component_name.lower()}_stability"] = float(loading_stability_frame[component_name].quantile(0.10))
            summary[f"min_{component_name.lower()}_stability"] = float(loading_stability_frame[component_name].min())
        else:
            summary[f"mean_{component_name.lower()}_stability"] = np.nan
            summary[f"median_{component_name.lower()}_stability"] = np.nan
            summary[f"p10_{component_name.lower()}_stability"] = np.nan
            summary[f"min_{component_name.lower()}_stability"] = np.nan

    return {
        "summary": summary,
        "rolling_pca": rolling_pca,
        "explained_variance_frame": explained_variance_frame,
        "loading_stability_frame": loading_stability_frame,
    }


# -------------------------------------------------------------------
# Regression layer from selected hedge tenors to PCA scores
# -------------------------------------------------------------------
def build_single_target_regression_model(
    regression_type: str,
    alpha_value: float,
    fit_intercept: bool,
    max_iterations: int,
):
    regression_type = regression_type.lower()

    if regression_type == "ols":
        return LinearRegression(fit_intercept=fit_intercept)

    if regression_type == "ridge":
        return Ridge(alpha=alpha_value, fit_intercept=fit_intercept)

    if regression_type == "lasso":
        return Lasso(alpha=alpha_value, fit_intercept=fit_intercept, max_iter=max_iterations)

    raise ValueError(f"Unknown regression type: {regression_type}")


def fit_pca_regression_on_selected_tenors(
    static_pca: StaticPCA,
    selected_tenors: List[str],
    regression_type: str = "ridge",
    alpha_value: float = 1.0,
    fit_intercept: bool = False,
    max_iterations: int = 20000,
) -> Dict[str, object]:
    processed_surface_moves = static_pca.data.processed_data.copy()
    missing_selected_tenors = [tenor for tenor in selected_tenors if tenor not in processed_surface_moves.columns]

    if missing_selected_tenors:
        raise ValueError(f"Selected tenors not found in processed data: {missing_selected_tenors}")

    selected_tenor_moves = processed_surface_moves[selected_tenors].copy()
    actual_scores = static_pca.results.scores.copy()
    pca_loadings = static_pca.results.loadings.copy()

    coefficient_rows = []
    intercept_values = {}
    predicted_scores_frame = pd.DataFrame(index=actual_scores.index, columns=actual_scores.columns, dtype=float)

    for pc_name in actual_scores.columns:
        regression_model = build_single_target_regression_model(
            regression_type=regression_type,
            alpha_value=alpha_value,
            fit_intercept=fit_intercept,
            max_iterations=max_iterations,
        )

        regression_model.fit(selected_tenor_moves.values, actual_scores[pc_name].values)

        coefficient_rows.append(pd.Series(regression_model.coef_, index=selected_tenors, name=pc_name))
        intercept_values[pc_name] = float(regression_model.intercept_) if fit_intercept else 0.0
        predicted_scores_frame[pc_name] = regression_model.predict(selected_tenor_moves.values)

    coefficient_frame = pd.concat(coefficient_rows, axis=1)
    intercept_series = pd.Series(intercept_values)

    reduced_matrix = coefficient_frame @ pca_loadings.T
    reconstructed_processed_surface = pd.DataFrame(
        predicted_scores_frame.values @ pca_loadings.values.T,
        index=predicted_scores_frame.index,
        columns=pca_loadings.index,
    )

    residual_processed_surface = processed_surface_moves.loc[
        reconstructed_processed_surface.index,
        reconstructed_processed_surface.columns
    ] - reconstructed_processed_surface

    surface_metrics = compute_reconstruction_metrics(
        actual_frame=processed_surface_moves.loc[
            reconstructed_processed_surface.index,
            reconstructed_processed_surface.columns
        ],
        residual_frame=residual_processed_surface,
    )

    score_fit_metrics = compute_score_fit_metrics(
        actual_scores=actual_scores,
        predicted_scores=predicted_scores_frame,
    )

    return {
        "selected_tenors": list(selected_tenors),
        "regression_type": regression_type,
        "alpha_value": alpha_value,
        "coefficients": coefficient_frame,
        "intercept": intercept_series,
        "predicted_scores": predicted_scores_frame,
        "reduced_matrix": reduced_matrix,
        "reconstructed_processed_surface": reconstructed_processed_surface,
        "actual_processed_surface": processed_surface_moves.loc[
            reconstructed_processed_surface.index,
            reconstructed_processed_surface.columns
        ],
        "residual_processed_surface": residual_processed_surface,
        "surface_metrics": surface_metrics,
        "score_fit_metrics": score_fit_metrics,
    }


def compute_matrix_turnover(matrix_sequence: List[pd.DataFrame]) -> pd.Series:
    turnover_rows = []

    for matrix_index in range(1, len(matrix_sequence)):
        previous_matrix = matrix_sequence[matrix_index - 1].values
        current_matrix = matrix_sequence[matrix_index].values

        numerator = float(np.linalg.norm(current_matrix - previous_matrix, ord="fro"))
        denominator = float(np.linalg.norm(previous_matrix, ord="fro"))
        turnover_value = numerator / denominator if denominator > 0 else np.nan
        turnover_rows.append(turnover_value)

    return pd.Series(turnover_rows, name="turnover")


def evaluate_selected_tenors_on_rolling_pca(
    rolling_pca: RollingPCA,
    selected_tenors: List[str],
    regression_type: str = "ridge",
    alpha_value: float = 1.0,
    fit_intercept: bool = False,
    max_iterations: int = 20000,
) -> Dict[str, object]:
    regression_results_by_window = []
    window_metric_rows = []
    all_rmse_by_date_series = []
    all_rmse_by_node_series = []
    all_coefficient_frames = []
    reduced_matrix_sequence = []
    coefficient_matrix_sequence = []

    for rolling_window_pca in rolling_pca.pcas_results:
        regression_result = fit_pca_regression_on_selected_tenors(
            static_pca=rolling_window_pca,
            selected_tenors=selected_tenors,
            regression_type=regression_type,
            alpha_value=alpha_value,
            fit_intercept=fit_intercept,
            max_iterations=max_iterations,
        )

        window_end_date = rolling_window_pca.results.scores.index[-1]
        surface_metrics = regression_result["surface_metrics"]
        score_fit_metrics = regression_result["score_fit_metrics"]

        window_metric_rows.append(
            {
                "window_end": window_end_date,
                "surface_r2": float(surface_metrics["R2"]),
                "surface_rmse": float(surface_metrics["RMSE Global"]),
                "surface_mae": float(surface_metrics["MAE Global"]),
                "score_mean_r2": float(score_fit_metrics["mean_r2"]),
                "score_median_r2": float(score_fit_metrics["median_r2"]),
            }
        )

        rmse_by_date = surface_metrics["RMSE by date"].copy()
        rmse_by_date.name = window_end_date
        all_rmse_by_date_series.append(rmse_by_date)

        rmse_by_node = surface_metrics["RMSE by node"].copy()
        rmse_by_node.name = window_end_date
        all_rmse_by_node_series.append(rmse_by_node)

        coefficient_frame = regression_result["coefficients"].copy()
        coefficient_frame.columns = pd.MultiIndex.from_product([[window_end_date], coefficient_frame.columns])
        all_coefficient_frames.append(coefficient_frame)

        reduced_matrix_sequence.append(regression_result["reduced_matrix"])
        coefficient_matrix_sequence.append(regression_result["coefficients"])

        regression_results_by_window.append(
            {
                "window_end": window_end_date,
                **regression_result,
            }
        )

    window_metrics_frame = pd.DataFrame(window_metric_rows).set_index("window_end")
    rmse_by_date_panel = pd.concat(all_rmse_by_date_series, axis=1)
    rmse_by_node_panel = pd.concat(all_rmse_by_node_series, axis=1)

    mean_rmse_by_node = rmse_by_node_panel.mean(axis=1).sort_index()
    median_rmse_by_node = rmse_by_node_panel.median(axis=1).sort_index()

    reduced_matrix_turnover = compute_matrix_turnover(reduced_matrix_sequence)
    coefficient_turnover = compute_matrix_turnover(coefficient_matrix_sequence)

    coefficient_history_panel = pd.concat(all_coefficient_frames, axis=1)

    summary = {
        "selected_tenors": list(selected_tenors),
        "subset_size": len(selected_tenors),
        "n_windows": len(regression_results_by_window),
        "mean_surface_r2": float(window_metrics_frame["surface_r2"].mean()),
        "median_surface_r2": float(window_metrics_frame["surface_r2"].median()),
        "p10_surface_r2": float(window_metrics_frame["surface_r2"].quantile(0.10)),
        "mean_surface_rmse": float(window_metrics_frame["surface_rmse"].mean()),
        "median_surface_rmse": float(window_metrics_frame["surface_rmse"].median()),
        "p95_surface_rmse": float(window_metrics_frame["surface_rmse"].quantile(0.95)),
        "mean_score_mean_r2": float(window_metrics_frame["score_mean_r2"].mean()),
        "median_score_mean_r2": float(window_metrics_frame["score_mean_r2"].median()),
        "p95_rmse_by_date": float(rmse_by_date_panel.stack().quantile(0.95)),
        "mean_reduced_matrix_turnover": float(reduced_matrix_turnover.mean()) if not reduced_matrix_turnover.empty else np.nan,
        "p95_reduced_matrix_turnover": float(reduced_matrix_turnover.quantile(0.95)) if not reduced_matrix_turnover.empty else np.nan,
        "mean_coefficient_turnover": float(coefficient_turnover.mean()) if not coefficient_turnover.empty else np.nan,
        "max_node_mean_rmse": float(mean_rmse_by_node.max()),
    }

    return {
        "summary": summary,
        "window_metrics": window_metrics_frame,
        "mean_rmse_by_node": mean_rmse_by_node,
        "median_rmse_by_node": median_rmse_by_node,
        "rmse_by_date_panel": rmse_by_date_panel,
        "rmse_by_node_panel": rmse_by_node_panel,
        "reduced_matrix_turnover": reduced_matrix_turnover,
        "coefficient_turnover": coefficient_turnover,
        "coefficient_history_panel": coefficient_history_panel,
        "regression_results_by_window": regression_results_by_window,
    }


# -------------------------------------------------------------------
# Greedy forward tenor selection
# -------------------------------------------------------------------
def rank_subset_results(result_frame: pd.DataFrame) -> pd.DataFrame:
    return result_frame.sort_values(
        by=[
            "median_surface_r2",
            "p95_rmse_by_date",
            "mean_reduced_matrix_turnover",
            "subset_size",
        ],
        ascending=[False, True, True, True],
    ).reset_index(drop=True)


def greedy_forward_tenor_selection(
    rolling_pca: RollingPCA,
    candidate_pool: List[str],
    subset_sizes_to_test: List[int],
    regression_type: str = "ridge",
    alpha_value: float = 1.0,
    fit_intercept: bool = False,
    max_iterations: int = 20000,
) -> Dict[str, object]:
    maximum_subset_size = max(subset_sizes_to_test)
    current_selected_tenors: List[str] = []

    step_history_rows = []
    best_result_by_subset_size: Dict[int, Dict[str, object]] = {}

    for target_subset_size in range(1, maximum_subset_size + 1):
        candidate_trial_rows = []
        detailed_trial_results: Dict[str, Dict[str, object]] = {}

        remaining_tenors = [tenor for tenor in candidate_pool if tenor not in current_selected_tenors]

        for tenor_to_add in remaining_tenors:
            trial_subset = current_selected_tenors + [tenor_to_add]

            trial_evaluation = evaluate_selected_tenors_on_rolling_pca(
                rolling_pca=rolling_pca,
                selected_tenors=trial_subset,
                regression_type=regression_type,
                alpha_value=alpha_value,
                fit_intercept=fit_intercept,
                max_iterations=max_iterations,
            )

            trial_key = " | ".join(trial_subset)
            detailed_trial_results[trial_key] = trial_evaluation

            candidate_trial_rows.append(
                {
                    "trial_subset": trial_subset,
                    "added_tenor": tenor_to_add,
                    **trial_evaluation["summary"],
                }
            )

        candidate_trial_frame = rank_subset_results(pd.DataFrame(candidate_trial_rows))
        best_trial_row = candidate_trial_frame.iloc[0]

        current_selected_tenors = list(best_trial_row["trial_subset"])

        step_history_rows.append(
            {
                "subset_size": target_subset_size,
                "selected_tenors": list(current_selected_tenors),
                "added_tenor": best_trial_row["added_tenor"],
                "median_surface_r2": best_trial_row["median_surface_r2"],
                "p95_rmse_by_date": best_trial_row["p95_rmse_by_date"],
                "mean_reduced_matrix_turnover": best_trial_row["mean_reduced_matrix_turnover"],
            }
        )

        if target_subset_size in subset_sizes_to_test:
            best_key = " | ".join(current_selected_tenors)
            best_result_by_subset_size[target_subset_size] = detailed_trial_results[best_key]

    subset_summary_rows = []

    for subset_size, evaluation_result in best_result_by_subset_size.items():
        subset_summary_rows.append(
            {
                "subset_size": subset_size,
                **evaluation_result["summary"],
            }
        )

    subset_summary_frame = rank_subset_results(pd.DataFrame(subset_summary_rows))
    step_history_frame = pd.DataFrame(step_history_rows)

    return {
        "step_history": step_history_frame,
        "subset_summary": subset_summary_frame,
        "best_result_by_subset_size": best_result_by_subset_size,
    }


# -------------------------------------------------------------------
# Visualization helpers
# -------------------------------------------------------------------
def plot_metric_heatmap_from_config_frame(
    configuration_frame: pd.DataFrame,
    metric_name: str,
    title: str,
    fmt: str = ".2f",
):
    pivot_frame = configuration_frame.pivot(index="window_size", columns="step_size", values=metric_name)

    plt.figure(figsize=(8, 5))
    sns.heatmap(pivot_frame, annot=True, fmt=fmt, cmap="viridis")
    plt.title(title)
    plt.ylabel("Window size")
    plt.xlabel("Step size")
    plt.show()


def reshape_node_metric_to_surface(metric_by_node: pd.Series) -> pd.DataFrame:
    surface_rows = []

    for node_label, metric_value in metric_by_node.items():
        expiry_label, tenor_label = parse_expiry_tenor_label(node_label)
        surface_rows.append(
            {
                "expiry": expiry_label,
                "tenor": tenor_label,
                "value": metric_value,
            }
        )

    surface_frame = pd.DataFrame(surface_rows)
    pivot_frame = surface_frame.pivot(index="expiry", columns="tenor", values="value")

    ordered_expiries = sort_maturity_labels(list(pivot_frame.index))
    ordered_tenors = sort_maturity_labels(list(pivot_frame.columns))

    return pivot_frame.loc[ordered_expiries, ordered_tenors]


def plot_surface_metric_heatmap(metric_by_node: pd.Series, title: str):
    pivot_frame = reshape_node_metric_to_surface(metric_by_node)

    plt.figure(figsize=(12, 7))
    sns.heatmap(pivot_frame, annot=True, fmt=".3f", cmap="magma")
    plt.title(title)
    plt.ylabel("Expiry")
    plt.xlabel("Underlying tenor")
    plt.show()


def plot_series_with_title(series_to_plot: pd.Series, title: str, y_label: str):
    plt.figure(figsize=(12, 5))
    plt.plot(series_to_plot.index, series_to_plot.values)
    plt.title(title)
    plt.ylabel(y_label)
    plt.xlabel("Date")
    plt.grid(True, alpha=0.3)
    plt.show()


def build_target_node_hedge_history(
    detailed_subset_evaluation: Dict[str, object],
    target_nodes: List[str],
) -> pd.DataFrame:
    hedge_history_rows = []

    for window_result in detailed_subset_evaluation["regression_results_by_window"]:
        window_end_date = window_result["window_end"]
        reduced_matrix = window_result["reduced_matrix"]

        for target_node in target_nodes:
            if target_node not in reduced_matrix.columns:
                continue

            for selected_tenor in reduced_matrix.index:
                hedge_history_rows.append(
                    {
                        "window_end": window_end_date,
                        "target_node": target_node,
                        "selected_tenor": selected_tenor,
                        "hedge_ratio": float(reduced_matrix.loc[selected_tenor, target_node]),
                    }
                )

    return pd.DataFrame(hedge_history_rows)


def plot_target_node_hedge_history(
    detailed_subset_evaluation: Dict[str, object],
    target_node: str,
):
    hedge_history_frame = build_target_node_hedge_history(
        detailed_subset_evaluation=detailed_subset_evaluation,
        target_nodes=[target_node],
    )

    hedge_history_frame = hedge_history_frame[hedge_history_frame["target_node"] == target_node]

    plt.figure(figsize=(12, 5))

    for selected_tenor, tenor_frame in hedge_history_frame.groupby("selected_tenor"):
        plt.plot(tenor_frame["window_end"], tenor_frame["hedge_ratio"], label=selected_tenor)

    plt.title(f"Hedge ratios across time for target node {target_node}")
    plt.xlabel("Window end")
    plt.ylabel("Hedge ratio")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def summarize_best_configuration(
    best_window_step_row: pd.Series,
    best_subset_row: pd.Series,
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "window_size": [best_window_step_row["window_size"]],
            "step_size": [best_window_step_row["step_size"]],
            "selected_tenors": [best_subset_row["selected_tenors"]],
            "median_surface_r2": [best_subset_row["median_surface_r2"]],
            "p95_rmse_by_date": [best_subset_row["p95_rmse_by_date"]],
            "mean_reduced_matrix_turnover": [best_subset_row["mean_reduced_matrix_turnover"]],
            "mean_topk_explained_variance": [best_window_step_row["mean_topk_explained_variance"]],
            "mean_pc1_stability": [best_window_step_row["mean_pc1_stability"]],
            "mean_pc2_stability": [best_window_step_row["mean_pc2_stability"]],
            "mean_pc3_stability": [best_window_step_row["mean_pc3_stability"]],
        }
    )

## Baseline static PCA sanity check

Before searching rolling configurations, it is worth checking that the global static PCA is economically sensible.

In [ ]:
covariance_estimator = CovarianceEstimator(
    method=covariance_method,
    halflife=ewma_halflife,
    alpha_ewma=alpha_ewma,
)

baseline_static_pca = StaticPCA(
    data=market_data,
    n_components=number_of_components,
    cov_estimator=covariance_estimator,
)
baseline_static_pca.fit()

explained_variance_percentages = baseline_static_pca.results.explained_var_ratios[:number_of_components] * 100.0
baseline_static_summary = pd.DataFrame(
    {
        "explained_variance_percent": explained_variance_percentages
    },
    index=[f"PC{component_index + 1}" for component_index in range(number_of_components)],
)

display(baseline_static_summary)

plt.figure(figsize=(10, 4))
plt.bar(baseline_static_summary.index, baseline_static_summary["explained_variance_percent"])
plt.title(f"Static PCA explained variance (Top {number_of_components})")
plt.ylabel("Explained variance (%)")
plt.show()

plt.figure(figsize=(16, 5))
for pc_name in baseline_static_pca.results.loadings.columns:
    plt.plot(baseline_static_pca.results.loadings.index, baseline_static_pca.results.loadings[pc_name], label=pc_name)
plt.axhline(0.0, linestyle="--", linewidth=1.0, color="black")
plt.title("Static PCA loadings")
plt.xticks(rotation=90)
plt.legend()
plt.show()

baseline_reconstructed_processed, baseline_actual_processed, baseline_residuals_processed = baseline_static_pca.reconstruct(
    revert_processing=False
)
baseline_processed_metrics = compute_reconstruction_metrics(
    actual_frame=baseline_actual_processed,
    residual_frame=baseline_residuals_processed
)

display(
    pd.DataFrame(
        {
            "RMSE Global": [baseline_processed_metrics["RMSE Global"]],
            "MAE Global": [baseline_processed_metrics["MAE Global"]],
            "R2": [baseline_processed_metrics["R2"]],
        }
    )
)

representative_nodes = [node for node in ["2Y10Y", "5Y10Y", "10Y10Y", "20Y20Y"] if node in baseline_actual_processed.columns]
plt.figure(figsize=(14, 6))
for representative_node in representative_nodes:
    plt.plot(baseline_actual_processed.index, baseline_actual_processed[representative_node], label=f"{representative_node} actual")
    plt.plot(
        baseline_reconstructed_processed.index,
        baseline_reconstructed_processed[representative_node],
        linestyle="--",
        label=f"{representative_node} reconstructed",
    )
plt.title("Static PCA reconstruction on processed moves")
plt.legend()
plt.show()

## Coarse search over rolling window length and rolling step

This section fits a rolling PCA for each `(window_size, step_size)` pair and measures:

- Top-K explained variance,
- PC loading stability across consecutive aligned windows.

In [ ]:
window_step_evaluation_results: Dict[Tuple[str, str], Dict[str, object]] = {}
window_step_summary_rows = []

for window_size in window_grid:
    for step_size in step_grid:
        print(f"Evaluating rolling PCA for window={window_size}, step={step_size} ...")

        evaluation_result = evaluate_window_step_configuration(
            base_market_data=market_data,
            window_size=window_size,
            step_size=step_size,
            covariance_estimator=covariance_estimator,
            number_of_components=number_of_components,
            alignment_anchor=alignment_anchor,
        )

        window_step_evaluation_results[(window_size, step_size)] = evaluation_result
        window_step_summary_rows.append(evaluation_result["summary"])

window_step_summary_frame = pd.DataFrame(window_step_summary_rows)
window_step_summary_frame = window_step_summary_frame.sort_values(
    by=[
        "mean_pc1_stability",
        "mean_pc2_stability",
        "p10_topk_explained_variance",
        "mean_topk_explained_variance",
    ],
    ascending=[False, False, False, False],
).reset_index(drop=True)

display(window_step_summary_frame)

## Heatmaps for the rolling PCA structural search

In [ ]:
plot_metric_heatmap_from_config_frame(
    configuration_frame=window_step_summary_frame,
    metric_name="mean_topk_explained_variance",
    title="Mean Top-K explained variance",
)

plot_metric_heatmap_from_config_frame(
    configuration_frame=window_step_summary_frame,
    metric_name="p10_topk_explained_variance",
    title="10th percentile Top-K explained variance",
)

plot_metric_heatmap_from_config_frame(
    configuration_frame=window_step_summary_frame,
    metric_name="mean_pc1_stability",
    title="Mean PC1 stability",
)

plot_metric_heatmap_from_config_frame(
    configuration_frame=window_step_summary_frame,
    metric_name="mean_pc2_stability",
    title="Mean PC2 stability",
)

plot_metric_heatmap_from_config_frame(
    configuration_frame=window_step_summary_frame,
    metric_name="mean_pc3_stability",
    title="Mean PC3 stability",
)

## Shortlist the best window / step pairs

This is a first admissibility filter. You can tighten or relax the thresholds depending on your tolerance for instability.

In [ ]:
admissible_window_step_frame = window_step_summary_frame[
    (window_step_summary_frame["mean_pc1_stability"] >= minimum_mean_pc1_stability)
    & (window_step_summary_frame["mean_pc2_stability"] >= minimum_mean_pc2_stability)
    & (window_step_summary_frame["p10_topk_explained_variance"] >= minimum_p10_topk_explained_variance)
].copy()

if admissible_window_step_frame.empty:
    print("No configuration passed the admissibility filter. Falling back to the top-ranked rows.")
    admissible_window_step_frame = window_step_summary_frame.copy()

shortlisted_window_step_frame = admissible_window_step_frame.head(number_of_window_step_pairs_to_shortlist).reset_index(drop=True)
display(shortlisted_window_step_frame)

## Optional diagnostics for shortlisted rolling PCA configurations

In [ ]:
for _, shortlisted_row in shortlisted_window_step_frame.iterrows():
    configuration_key = (shortlisted_row["window_size"], shortlisted_row["step_size"])
    shortlisted_result = window_step_evaluation_results[configuration_key]

    print(f"Diagnostics for window={configuration_key[0]}, step={configuration_key[1]}")
    display(shortlisted_result["explained_variance_frame"].tail())
    display(shortlisted_result["loading_stability_frame"].tail())

    plt.figure(figsize=(12, 4))
    plt.plot(shortlisted_result["explained_variance_frame"].index, shortlisted_result["explained_variance_frame"]["TopK"])
    plt.title(f"Top-K explained variance through time | window={configuration_key[0]} | step={configuration_key[1]}")
    plt.ylabel("Explained variance (%)")
    plt.grid(True, alpha=0.3)
    plt.show()

    if not shortlisted_result["loading_stability_frame"].empty:
        plt.figure(figsize=(12, 4))
        for component_name in shortlisted_result["loading_stability_frame"].columns:
            plt.plot(
                shortlisted_result["loading_stability_frame"].index,
                shortlisted_result["loading_stability_frame"][component_name],
                label=component_name,
            )
        plt.title(f"PC stability through time | window={configuration_key[0]} | step={configuration_key[1]}")
        plt.ylabel("Cosine similarity")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

## Greedy forward search for the best hedge-tenor subset

For each shortlisted rolling PCA configuration:

- start with an empty set,
- add the tenor that improves the tradable replication the most,
- continue until the largest requested subset size is reached.

The ranking logic favors:

1. higher median surface R²,
2. lower downside date-level RMSE,
3. lower hedge-mapping turnover,
4. smaller subsets.

In [ ]:
subset_search_results_by_configuration: Dict[Tuple[str, str], Dict[str, object]] = {}
subset_summary_rows = []

for _, shortlisted_row in shortlisted_window_step_frame.iterrows():
    configuration_key = (shortlisted_row["window_size"], shortlisted_row["step_size"])
    rolling_pca = window_step_evaluation_results[configuration_key]["rolling_pca"]

    print(f"Running greedy forward selection for window={configuration_key[0]}, step={configuration_key[1]} ...")

    subset_search_result = greedy_forward_tenor_selection(
        rolling_pca=rolling_pca,
        candidate_pool=candidate_pool,
        subset_sizes_to_test=subset_sizes_to_test,
        regression_type=regression_type,
        alpha_value=ridge_or_lasso_alpha,
        fit_intercept=fit_intercept,
        max_iterations=lasso_max_iterations,
    )

    subset_search_results_by_configuration[configuration_key] = subset_search_result

    subset_summary_frame_for_configuration = subset_search_result["subset_summary"].copy()
    subset_summary_frame_for_configuration["window_size"] = configuration_key[0]
    subset_summary_frame_for_configuration["step_size"] = configuration_key[1]

    subset_summary_rows.append(subset_summary_frame_for_configuration)

all_subset_summary_frame = pd.concat(subset_summary_rows, axis=0, ignore_index=True)
all_subset_summary_frame = all_subset_summary_frame.sort_values(
    by=[
        "median_surface_r2",
        "p95_rmse_by_date",
        "mean_reduced_matrix_turnover",
        "subset_size",
    ],
    ascending=[False, True, True, True],
).reset_index(drop=True)

display(all_subset_summary_frame)

## Inspect the greedy forward path

In [ ]:
for configuration_key, subset_search_result in subset_search_results_by_configuration.items():
    print(f"Greedy path for window={configuration_key[0]}, step={configuration_key[1]}")
    display(subset_search_result["step_history"])

## Choose the best final configuration

This cell takes the top row according to the notebook ranking logic.
You can of course override it manually if desk constraints matter more than the ranking.

In [ ]:
best_subset_row = all_subset_summary_frame.iloc[0]
best_window_size = best_subset_row["window_size"]
best_step_size = best_subset_row["step_size"]
best_subset_size = int(best_subset_row["subset_size"])

best_window_step_row = shortlisted_window_step_frame[
    (shortlisted_window_step_frame["window_size"] == best_window_size)
    & (shortlisted_window_step_frame["step_size"] == best_step_size)
].iloc[0]

best_detailed_subset_result = subset_search_results_by_configuration[
    (best_window_size, best_step_size)
]["best_result_by_subset_size"][best_subset_size]

final_selection_summary = summarize_best_configuration(
    best_window_step_row=best_window_step_row,
    best_subset_row=best_subset_row,
)

display(final_selection_summary)

## Diagnostics for the best final configuration

In [ ]:
print("Best window size:", best_window_size)
print("Best step size:", best_step_size)
print("Best selected tenors:", best_subset_row["selected_tenors"])

display(best_detailed_subset_result["window_metrics"].head())
display(best_detailed_subset_result["window_metrics"].tail())

plot_surface_metric_heatmap(
    metric_by_node=best_detailed_subset_result["mean_rmse_by_node"],
    title="Average RMSE by node for the best selected hedge subset",
)

if not best_detailed_subset_result["reduced_matrix_turnover"].empty:
    plot_series_with_title(
        series_to_plot=best_detailed_subset_result["reduced_matrix_turnover"],
        title="Reduced hedge-mapping turnover across rolling windows",
        y_label="Turnover",
    )

if not best_detailed_subset_result["coefficient_turnover"].empty:
    plot_series_with_title(
        series_to_plot=best_detailed_subset_result["coefficient_turnover"],
        title="Regression coefficient turnover across rolling windows",
        y_label="Turnover",
    )

## Coefficient history for the best selected hedge tenors

This is useful to see whether the regression is stable tenor by tenor and PC by PC.

In [ ]:
coefficient_history_panel = best_detailed_subset_result["coefficient_history_panel"]

for pc_name in [f"PC{component_index + 1}" for component_index in range(number_of_components)]:
    plt.figure(figsize=(12, 5))

    for selected_tenor in best_subset_row["selected_tenors"]:
        coefficient_series = coefficient_history_panel.loc[selected_tenor].xs(pc_name, level=1)
        plt.plot(coefficient_series.index, coefficient_series.values, label=selected_tenor)

    plt.title(f"Regression coefficients across time for {pc_name}")
    plt.xlabel("Window end")
    plt.ylabel("Coefficient")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## Hedge-ratio history for representative target nodes

These are not desk sensitivities yet.  
They are simply the node-level hedge ratios implied by the reduced mapping.

In [ ]:
representative_target_nodes = [node for node in ["2Y10Y", "5Y10Y", "10Y10Y", "20Y20Y"] if node in market_data.processed_data.columns]

for representative_target_node in representative_target_nodes:
    plot_target_node_hedge_history(
        detailed_subset_evaluation=best_detailed_subset_result,
        target_node=representative_target_node,
    )

## Final ranking tables to export if needed

In [ ]:
window_step_ranking_output = window_step_summary_frame.copy()
subset_ranking_output = all_subset_summary_frame.copy()

display(window_step_ranking_output)
display(subset_ranking_output)

## Notes for the next iteration

Once this notebook has identified a good region of the parameter space, the next steps should be:

1. repeat the same search **pseudo out-of-sample**,
2. evaluate the reduced mapping on **synthetic books**,
3. replace the synthetic loss function with **real desk sensitivity PnL** when available,
4. then compare this PCA + regression framework with your future **implied sparse PCA regression** framework.